# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### The method: a classifier's probability, ranked and cut at K

My lane is **ranking** ("which pages first?"), not classification, and I settled that in ML-03. But `training-honest-models` maps ranking to *"any classifier's probability, evaluated at precision@K"*, because a ranking needs a continuous score and a classifier's `predict_proba` is exactly that. So I fit classifiers and rank by probability. The task stays ranking; the classifier is the mechanism.

### Why these four, in this order

| Model | Why it is in the comparison |
|---|---|
| **Logistic Regression** | The readable one. If a linear combination already beats my rule, that is the honest answer and everything heavier is decoration. |
| **Decision Tree (depth 3)** | Printable. A tree I can read end-to-end is worth more than an opaque model a point or two stronger, and it tells me *which thresholds* matter. |
| **Random Forest** | The first model here that can express interactions. ML-07 found the flag was good but the *ordering inside it* was wrong; that is an interaction between page size and CTR shortfall, which a forest can learn and a linear model cannot. |
| **Gradient Boosting** | Included specifically to test whether more complexity still pays. If it does not beat the forest, that is a result worth reporting, not a failure to hide. |

I also include a **base-rate row** (the majority-class floor) so every number has something to be measured against.

### What I am not doing

No clustering: I have an observed label and a ranking decision, so an unsupervised method would answer a question nobody asked. No hyperparameter search either: with 32 clients and split-to-split noise around ±0.12 (measured in ML-07 and again below), tuning would be fitting noise, and I could not tell a real gain from a lucky split.

**Simplicity is the tie-breaker.** Where two models sit within the split-to-split spread of each other, I report them as tied and prefer the simpler one, rather than crowning whichever mean happens to be higher.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Grouped by client, and repeated 8 times

**Grouped, not random.** Pages from one client share a template, an editorial calendar, and a niche. A random row split would put a client's pages in both train and test, and the model would score well by recognising the *client* rather than learning what a declining page looks like. The decision this supports is "here is a queue for a site we may not have modelled before", so the test fold must contain clients the model has never seen. `GroupShuffleSplit` on `client_id`, 25% of clients held out.

**Repeated, not once.** In ML-07 a single split moved precision@50 by up to 0.14, enough to reverse which method looked better. With only 32 clients, one split is a coin toss, so I run **8 different client splits (seeds 0–7)** and report the mean with its spread. Any difference smaller than that spread is not a difference.

**Paired, because the comparison deserves it.** Every model and the baseline see *the same 8 splits*. That lets me compare them split-by-split rather than mean-vs-mean, which is a stricter test: I can ask "in how many splits did the model actually win?" instead of trusting two averages that overlap.

### One honest note about the baseline number

My ML-07 baseline reported precision@50 = 0.500 on the **whole** dataset with no split. It cannot be compared directly to a number computed on held-out clients, because those are different rows with a different base rate. So I **re-run the same frozen rule inside this notebook**, on these test folds, and that re-run is what appears in the table below. It scores higher here (0.605) than the ML-07 headline, and that is a change in the evaluation, not a change in the rule.

One detail to keep it honest: the rule's position-band CTR medians are computed on the **training** clients only and applied to the test fold. Fitting them on all the data would leak test information into the baseline and quietly flatter it.

### Leakage guard

`trend_direction` and `trend_pct` are excluded, because the label is derived from them. So are `impressions_last_30d` and `impressions_prev_30d`: the data dictionary defines `trend_pct` as exactly `(last_30d − prev_30d) / prev_30d`, so a model holding both columns can reconstruct the label outright. The next cell asserts this rather than trusting it.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd, sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

SEED, N_SPLITS, K = 42, 8, 50          # seeds fixed; see reproducibility note below
if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])          # traffic is heavy-tailed

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate",
       "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "age_tier",
       "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

# --- LEAKAGE GUARD: assert, don't assume ------------------------------------
BANNED = {"trend_direction", "trend_pct",                      # label is derived from these
          "impressions_last_30d", "impressions_prev_30d",      # trend_pct IS their ratio
          "clicks_last_30d", "clicks_prev_30d",
          "sessions_last_30d", "sessions_prev_30d"}
assert not (set(NUM + CAT) & BANNED), f"LEAK: {set(NUM+CAT) & BANNED}"
print(f"features: {len(NUM)} numeric + {len(CAT)} categorical | banned columns excluded: OK")

def precision_at_k(scores, labels, k=K):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

def baseline_rule(train, test):
    """My FROZEN ML-07 rule. Band medians fitted on TRAIN only, applied to TEST."""
    def prep(d):
        d = d.copy()
        d["visible"] = d["impressions_90d"] >= 500
        d["has_pos"] = d["avg_position"] > 0            # 0 = no data, not rank zero
        d["top20"] = d["has_pos"] & (d["avg_position"] <= 20)
        d["pos_band"] = pd.cut(d["avg_position"].where(d["has_pos"]),
                               [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
        return d
    tr, te = prep(train), prep(test)
    med = tr.groupby("pos_band", observed=True)["ctr"].median()      # TRAIN only
    bm = te["pos_band"].map(med).astype(float)
    shortfall = (bm - te["ctr"]).clip(lower=0).fillna(0)
    w = np.where(~te["visible"], 0,
        np.where(te["top20"] & (te["ctr"] < bm), 3,
        np.where(te["freshness_tier"].eq("91-180"), 2, 1)))
    return w * np.log10(te["impressions_90d"].clip(lower=1)) * (1 + shortfall)

def pipe(model):
    return Pipeline([("prep", ColumnTransformer([
        ("n", Pipeline([("i", SimpleImputer(strategy="median")),
                        ("s", StandardScaler())]), NUM),
        ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CAT),
    ])), ("m", model)])

MODELS = {
    "logistic_regression": lambda: pipe(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
    "decision_tree_d3":    lambda: pipe(DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED)),
    "random_forest":       lambda: pipe(RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                        class_weight="balanced", n_jobs=-1, random_state=SEED)),
    "gradient_boosting":   lambda: pipe(GradientBoostingClassifier(random_state=SEED)),
}

y = df["is_declining_label"].values
rows, t0 = [], time.time()
for seed in range(N_SPLITS):
    tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                      .split(df, y, groups=df["client_id"]))
    train, test, yte = df.iloc[tr_i], df.iloc[te_i], y[te_i]
    rows.append(dict(seed=seed, model="base rate (floor)", p20=yte.mean(), p50=yte.mean(),
                     auc=0.500, ap=yte.mean()))
    bs = baseline_rule(train, test)
    rows.append(dict(seed=seed, model="baseline rule (ML-07)", p20=precision_at_k(bs, yte, 20),
                     p50=precision_at_k(bs, yte), auc=roc_auc_score(yte, bs),
                     ap=average_precision_score(yte, bs)))
    for name, make in MODELS.items():
        prob = make().fit(train[NUM + CAT], y[tr_i]).predict_proba(test[NUM + CAT])[:, 1]
        rows.append(dict(seed=seed, model=name, p20=precision_at_k(prob, yte, 20),
                         p50=precision_at_k(prob, yte), auc=roc_auc_score(yte, prob),
                         ap=average_precision_score(yte, prob)))
print(f"{N_SPLITS} client-grouped splits x {len(MODELS)} models  [{time.time()-t0:.0f}s]\n")

res = pd.DataFrame(rows)
ORDER = ["base rate (floor)", "baseline rule (ML-07)", "logistic_regression",
         "decision_tree_d3", "random_forest", "gradient_boosting"]
table = (res.groupby("model").agg(**{
    "P@20": ("p20", "mean"), "P@20 sd": ("p20", "std"),
    "P@50": ("p50", "mean"), "P@50 sd": ("p50", "std"),
    "ROC-AUC": ("auc", "mean"), "AvgPrec": ("ap", "mean")})
    .reindex(ORDER).round(3))

print("=" * 82)
print("MODEL vs BASELINE  -- same data, same 8 client-grouped splits, same metric")
print("=" * 82)
print(table.to_string())

# --- paired comparison: same splits, so compare split-by-split ---------------
P = res.pivot(index="seed", columns="model", values="p50")
print("\nPAIRED differences in P@50 (stricter than comparing two means):")
for a, b in [("random_forest", "baseline rule (ML-07)"),
             ("logistic_regression", "baseline rule (ML-07)"),
             ("random_forest", "logistic_regression"),
             ("random_forest", "gradient_boosting")]:
    d = P[a] - P[b]
    print(f"  {a:<20} - {b:<22} {d.mean():+.3f} (sd {d.std():.3f})  "
          f"wins {int((d > 0).sum())}/{N_SPLITS}  worst split {d.min():+.3f}")


features: 18 numeric + 8 categorical | banned columns excluded: OK


8 client-grouped splits x 4 models  [85s]

MODEL vs BASELINE  -- same data, same 8 client-grouped splits, same metric
                        P@20  P@20 sd   P@50  P@50 sd  ROC-AUC  AvgPrec
model                                                                  
base rate (floor)      0.495    0.068  0.495    0.068    0.500    0.495
baseline rule (ML-07)  0.531    0.144  0.605    0.114    0.583    0.566
logistic_regression    0.738    0.160  0.735    0.120    0.673    0.640
decision_tree_d3       0.606    0.164  0.622    0.126    0.657    0.600
random_forest          0.856    0.094  0.805    0.108    0.710    0.676
gradient_boosting      0.794    0.178  0.782    0.146    0.712    0.675

PAIRED differences in P@50 (stricter than comparing two means):
  random_forest        - baseline rule (ML-07)  +0.200 (sd 0.184)  wins 7/8  worst split -0.200
  logistic_regression  - baseline rule (ML-07)  +0.130 (sd 0.141)  wins 7/8  worst split -0.080
  random_forest        - logistic_regression    +

### Reading the table

**The model beats the rule, and the win survives repetition.** Random forest reaches precision@50 ≈ 0.805 against the frozen baseline's 0.605 and a 0.495 base rate. The paired test is the one I trust: the forest wins in **7 of 8 client splits**, average **+0.200**. That is a real improvement, not a lucky fold.

**Complexity stops paying almost immediately.** Gradient boosting, the heaviest model here, wins only **5 of 8** splits against the forest, average **+0.022** with a spread of 0.118. That is a coin flip. And plain **logistic regression** already delivers most of the gain: +0.130 over the baseline, winning 7 of 8. So the honest ordering is *rule → logistic regression* is the big step; *logistic regression → forest* is a smaller real gain (6/8, +0.070); *forest → boosting* is nothing at all.

If someone needs a model they can explain to a client, logistic regression is defensible and gives up little. I would not ship gradient boosting on this evidence.

### What I am not claiming

The spreads are wide, ±0.11 to ±0.15 on P@50, because 32 clients is a small panel and each split holds out only 8 of them. So I quote "roughly 0.80, ±0.11", never 0.805. **Split 3 is the honest counter-example: the baseline scored 0.80 there and the forest only 0.60.** One split in eight, the hand-written rule won outright. That is what a 7/8 win rate actually looks like from the inside, and it is why I report the win *count* rather than only the mean.

### One methodological note

The baseline's re-run number (0.605) is higher than the 0.500 I published in ML-07: same frozen rule, different evaluation. ML-07 scored the top 50 of all 30,000 rows; here it scores the top 50 of a ~7,500-row held-out fold with a different base rate. The ML-07 figure and this one are not comparable, and only the numbers inside this table should be compared with each other.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [2]:
from sklearn.inspection import permutation_importance

# One representative split (seed 0) for interpretation.
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                  .split(df, y, groups=df["client_id"]))
train, test, yte = df.iloc[tr_i], df.iloc[te_i], y[te_i]
rf = MODELS["random_forest"]().fit(train[NUM + CAT], y[tr_i])
prob = rf.predict_proba(test[NUM + CAT])[:, 1]
print(f"held-out clients: {test['client_id'].nunique()} | rows: {len(test):,} "
      f"| test base rate: {yte.mean():.3f}\n")

# --- WHAT IT LEANS ON: permutation importance, measured on held-out clients --
# Permutation, not impurity: impurity importance inflates high-cardinality features.
pi = permutation_importance(rf, test[NUM + CAT], yte, n_repeats=5,
                            random_state=SEED, scoring="roc_auc", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=NUM + CAT).sort_values(ascending=False)
print("PERMUTATION IMPORTANCE - drop in ROC-AUC when the column is shuffled (top 8):")
print(imp.head(8).round(4).to_string())
print("\nSanity check: the top feature costs only "
      f"{imp.iloc[0]:.3f} AUC when shuffled. No single column carries the answer,")
print("which is what I want to see -- a suspiciously dominant feature usually means leakage.")

# --- WHERE IT IS WRONG ------------------------------------------------------
t = test.copy(); t["prob"] = prob; t["label"] = yte
t["pred"] = (t["prob"] >= 0.5).astype(int)
t["correct"] = t["pred"] == t["label"]

print("\n" + "=" * 60)
print("ERRORS BY SEGMENT")
print("=" * 60)
print("by impression tier:")
print(t.groupby("impression_tier", observed=True)["correct"]
       .agg(n="size", accuracy="mean").sort_values("accuracy").round(3).to_string())
print("\nby position tier:")
print(t.groupby("position_tier", observed=True)["correct"]
       .agg(n="size", accuracy="mean").sort_values("accuracy").round(3).to_string())
print("\nworst 3 held-out clients:")
print(t.groupby("client_id")["correct"].agg(n="size", accuracy="mean")
       .sort_values("accuracy").head(3).round(3).to_string())

# --- THREE CONCRETE WRONG CASES from the actual queue -----------------------
print("\n" + "=" * 60)
print("3 MOST CONFIDENT MISTAKES INSIDE THE TOP-50 QUEUE")
print("(these are the rows that would waste an editor's week)")
print("=" * 60)
top50 = t.nlargest(50, "prob")
worst = top50[top50["label"] == 0].nlargest(3, "prob")
print(worst[["prob", "impressions_90d", "avg_position", "ctr", "days_since_last_update",
             "content_type", "trend_direction"]].to_string(index=False))
print(f"\ntop-50 queue precision on this split: {top50['label'].mean():.2f} "
      f"({int(top50['label'].sum())}/50 genuinely declining)")


held-out clients: 8 | rows: 11,285 | test base rate: 0.492



PERMUTATION IMPORTANCE - drop in ROC-AUC when the column is shuffled (top 8):
days_with_impressions    0.0679
avg_position             0.0407
content_age_days         0.0206
log_impressions_90d      0.0201
position_tier            0.0137
age_tier                 0.0131
impression_tier          0.0053
freshness_tier           0.0043

Sanity check: the top feature costs only 0.068 AUC when shuffled. No single column carries the answer,
which is what I want to see -- a suspiciously dominant feature usually means leakage.

ERRORS BY SEGMENT
by impression tier:
                    n  accuracy
impression_tier                
excellent         529     0.569
good             2810     0.585
moderate         3888     0.668
low              4058     0.725

by position tier:
                  n  accuracy
position_tier                
striking       2247     0.606
page_1         5330     0.617
page_3_5       2008     0.699
deep            460     0.733
top_3          1240     0.883

worst 3 held-ou

### What the model leans on

The top three by permutation importance, and whether each makes sense:

1. **`days_with_impressions`** (−0.068 AUC when shuffled): how many of the 90 days the page appeared in search at all. A page slipping from daily visibility to occasional is losing ground *before* the traffic total shows it. This is the most sensible top feature I could have hoped for, and notably it is **not** something my hand rule used.
2. **`avg_position`** (−0.041): where it ranks. Expected, and it is why the rule bothered with position bands.
3. **`content_age_days`** (−0.021): older pages decline more often. This is the confounder I flagged as risk #1 back in ML-02: age may be driving both "stale" and "declining". It matters, but it ranks *third* and costs little when shuffled, so the model is not merely an age detector. That specific worry is now partly retired.

Shuffling the strongest feature costs only 0.068 AUC. Nothing dominates, which is the reassuring shape. A feature that cost 0.30 would have me hunting for a leak.

### Where it is wrong

**It is worst on exactly the pages an editor cares most about.** Accuracy is 0.569 on `excellent`-tier impression pages and 0.585 on `good`, versus 0.725 on `low`. The model is most reliable about pages with the least traffic at stake. This is the *same* weakness I measured in the ML-07 baseline, which means it is a property of the data, not of either method: big pages behave differently and there are few of them to learn from.

By position, accuracy is 0.883 in `top_3` but only 0.606 for `striking` and 0.617 for `page_1`. The crowded middle, where small ranking movements swing traffic, is genuinely harder.

**One held-out client scores 0.445, below a coin flip.** With 8 clients per test fold, a single unusual client visibly drags the fold, which is exactly why one split is not a result.

### Three concrete mistakes, and why they are hard

The three most confident false positives inside the top-50 queue share one shape: **`keyword article` pages with CTR of exactly 0.00 and small impression counts (206–612), whose real trend was `up` or `stable`.**

The model reads "impressions but zero clicks" as a page decaying. For a small page that is often the opposite: it is a *new or rising* page that has started appearing in search and has not yet accumulated clicks. Zero clicks over 90 days is ambiguous: it can mean decline, or it can mean the page has barely started. Nothing in the features distinguishes "fading" from "just arrived", because both look like low clicks on modest impressions.

That is a fixable gap, and it points at the next piece of work: this dataset is a single 90-day snapshot with no history, so "direction of travel before now" is unknowable. The warehouse daily table from ML-04 *does* have real dates, so a "days since first impression" feature would separate a new page from a dying one, and I would expect it to remove most of this error class.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere: pseudonymous IDs only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


